# Interactive plots: Acceleration RMS vs TIMESTAMP

This notebook scans `data/*.csv` and plots only `Acceleration RMS` against `TIMESTAMP` for each CSV.

Zoom and pan are enabled via `%matplotlib notebook`.

In [ ]:
%matplotlib notebook

from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

print('Jupyter cwd:', os.getcwd(), flush=True)

DATA_DIR = Path('data')
print('DATA_DIR exists:', DATA_DIR.exists(), flush=True)

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) in {DATA_DIR.resolve()}', flush=True)

for f in csv_files:
    print('-', f.name, flush=True)


In [ ]:
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

required_cols = {'TIMESTAMP', 'Acceleration RMS'}

for csv_path in csv_files:
    print(f'Plotting: {csv_path.name}', flush=True)
    df = pd.read_csv(csv_path)

    missing = required_cols - set(df.columns)
    if missing:
        print(f'  Skipping {csv_path.name}: missing columns {sorted(missing)}', flush=True)
        continue

    ts = pd.to_datetime(df['TIMESTAMP'], errors='coerce', dayfirst=True)
    y = pd.to_numeric(df['Acceleration RMS'], errors='coerce')
    valid = ts.notna() & y.notna()

    if valid.sum() == 0:
        print(f'  Skipping {csv_path.name}: no valid TIMESTAMP/Acceleration RMS rows', flush=True)
        continue

    ts = ts[valid]
    y = y[valid]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts, y, linewidth=1.0)
    ax.set_title(f'{csv_path.name} - Acceleration RMS vs TIMESTAMP')
    ax.set_xlabel('TIMESTAMP')
    ax.set_ylabel('Acceleration RMS')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [5]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path

# Open interactive plots in browser
pio.renderers.default = "browser"

# ============================================
# Folder containing CSV files
# ============================================
folder_path = r"data/"

# Get all CSV files
csv_files = list(Path(folder_path).glob("*.csv"))

# ============================================
# Plot each CSV separately
# ============================================
for file in csv_files:

    print(f"Plotting: {file.name}")

    # Read CSV
    df = pd.read_csv(file)

    # Convert timestamp column
    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"],
        dayfirst=True,
        format="mixed"
    )

    # Create figure
    fig = px.line(
        df,
        x="TIMESTAMP",
        y="Acceleration RMS",
        title=file.stem
    )

    # Layout
    fig.update_layout(
        xaxis_title="Timestamp",
        yaxis_title="Acceleration RMS",
        hovermode="x unified",
        template="plotly_white"
    )

    # Show separate interactive plot
    fig.show()

Plotting: AHU 2-9 Blower DE A.csv
Plotting: AHU 2-9 Blower DE V.csv
Plotting: AHU 2-9 Blower DE Vibration X.csv
Plotting: AHU 2-9 Blower NDE A.csv
Plotting: AHU 2-9 Blower NDE H.csv
Plotting: AHU 2-9 Blower NDE V.csv
Plotting: AHU 2-9 motor DE H.csv
Plotting: AHU 2-9 motor NDE H.csv


In [9]:
import pandas as pd

# ============================================
# Load CSV
# ============================================
file_path = r"data/AHU 2-9 motor NDE H.csv"

df = pd.read_csv(file_path)

# ============================================
# Convert timestamp column
# ============================================
df["TIMESTAMP"] = pd.to_datetime(
    df["TIMESTAMP"],
    dayfirst=True,
    format="mixed"
)

# ============================================
# Sort chronologically
# ============================================
df = df.sort_values("TIMESTAMP")

# ============================================
# Split 80% / 20%
# ============================================
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# ============================================
# Save split CSVs
# ============================================
train_df.to_csv("data_train_val.csv", index=False)
test_df.to_csv("data_test_anomalous.csv", index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain period:")
print(train_df["TIMESTAMP"].min(), "->", train_df["TIMESTAMP"].max())

print("\nTest period:")
print(test_df["TIMESTAMP"].min(), "->", test_df["TIMESTAMP"].max())

Train shape: (58721, 17)
Test shape: (14681, 17)

Train period:
2023-08-25 16:39:00 -> 2025-02-26 18:41:00

Test period:
2025-02-26 18:46:00 -> 2026-01-20 10:19:00
